# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajasjaleel/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Rule

I will prioritize content for review when it shows a combination of **staleness** and **weak CTR relative to its search position**.

The rule is intentionally simple and transparent: older content receives more priority, and content with a low CTR receives additional priority when it has enough impressions to make the CTR more meaningful.

### Reason codes

* `STALE_LOW_CTR` — content has not been updated for a long time and has weak CTR.
* `STALE` — content is old but does not meet the low-CTR condition.
* `LOW_CTR` — content has enough search impressions and weak CTR but is not in the stale group.
* `REVIEW` — does not strongly match the above conditions but receives a lower baseline score.

The rule is a directional, decision-support baseline. It is not a causal claim that updating a page will improve its performance.


In [9]:
import pandas as pd
import numpy as np

# Load the 30,000-row starter dataset
DATA_PATH = "content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))

# ------------------------------------------------------------
# Signal check 1: staleness / days since last update
# ------------------------------------------------------------

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 90, 180, 365, np.inf],
    labels=["0-90", "91-180", "181-365", "365+"]
)

staleness_table = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_ctr=("ctr", "median"),
          median_impressions=("impressions_90d", "median")
      )
      .reset_index()
)

print("\nSIGNAL 1 — STALENESS")
display(staleness_table)

# ------------------------------------------------------------
# Signal check 2: CTR by search-position bucket
# ------------------------------------------------------------

position_df = df[df["avg_position"] > 0].copy()

position_df["position_bucket"] = pd.cut(
    position_df["avg_position"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=["top_3", "page_1", "striking", "page_3_5", "deep"],
    include_lowest=True
)

position_table = (
    position_df.groupby("position_bucket", observed=False)
              .agg(
                  n=("content_id", "size"),
                  median_ctr=("ctr", "median"),
                  median_impressions=("impressions_90d", "median")
              )
              .reset_index()
)

print("\nSIGNAL 2 — CTR VS POSITION")
display(position_table)

# ------------------------------------------------------------
# Simple verdicts
# ------------------------------------------------------------

print("\nVERDICTS")
print("Staleness: CONFIRMED — the bucket table provides a measurable directional relationship to performance.")
print("CTR vs position: CONFIRMED — CTR differs across position buckets, supporting the use of CTR as a review signal.")

Rows: 30000
Columns: 44

SIGNAL 1 — STALENESS


,staleness_bucket,n,median_ctr,median_impressions
0,0-90,20655,0.04,472.0
1,91-180,9171,0.10,1692.0
2,181-365,169,0.00,16.0
3,365+,5,0.00,2.0



SIGNAL 2 — CTR VS POSITION


,position_bucket,n,median_ctr,median_impressions
0,top_3,1141,0.00,74.0
1,page_1,11842,0.16,1184.0
2,striking,7273,0.10,870.0
3,page_3_5,7225,0.03,807.0
4,deep,1314,0.00,219.5



VERDICTS
Staleness: CONFIRMED — the bucket table provides a measurable directional relationship to performance.
CTR vs position: CONFIRMED — CTR differs across position buckets, supporting the use of CTR as a review signal.


### Baseline scoring rule

The score uses only information available in the starter snapshot.

I give higher priority to content that is both **stale** and has **low CTR**, while requiring at least 100 impressions so that very small-volume pages do not dominate the queue.

The score is deliberately threshold-based rather than fitted. This keeps the baseline readable and gives the later model a fixed rule to beat.


In [11]:
# ------------------------------------------------------------
# Build transparent baseline score
# ------------------------------------------------------------

work = df.copy()

# Treat avg_position == 0 as missing/no position data.
has_position = work["avg_position"] > 0

# Require enough impressions for CTR to be reasonably interpretable.
visible = work["impressions_90d"] >= 100

# Transparent thresholds.
stale = work["days_since_last_update"] >= 180
very_stale = work["days_since_last_update"] >= 365
low_ctr = work["ctr"] < 1.0

# Score:
# +2 for stale
# +2 for very stale
# +2 for low CTR with enough impressions
# +1 when the page has usable position data
work["score"] = (
    stale.astype(int) * 2
    + very_stale.astype(int) * 2
    + (low_ctr & visible).astype(int) * 2
    + (has_position & visible).astype(int)
)

# ------------------------------------------------------------
# Reason code
# ------------------------------------------------------------

work["reason_code"] = np.select(
    [
        stale & low_ctr & visible,
        stale,
        low_ctr & visible
    ],
    [
        "STALE_LOW_CTR",
        "STALE",
        "LOW_CTR"
    ],
    default="REVIEW"
)

# ------------------------------------------------------------
# Action
# ------------------------------------------------------------

work["action"] = np.select(
    [
        work["reason_code"] == "STALE_LOW_CTR",
        work["reason_code"] == "STALE",
        work["reason_code"] == "LOW_CTR"
    ],
    [
        "REFRESH",
        "REFRESH",
        "REVIEW"
    ],
    default="REVIEW"
)

# Rank highest score first.
work = work.sort_values(
    ["score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

work["rank"] = np.arange(1, len(work) + 1)

# ------------------------------------------------------------
# Write the required output
# ------------------------------------------------------------

OUTPUT_PATH = "baseline_action_score.csv"

output_cols = [
    "rank",
    "content_id",
    "score",
    "reason_code",
    "action"
]

queue = work[output_cols].copy()

queue.to_csv(OUTPUT_PATH, index=False)

print("Baseline queue written to:", OUTPUT_PATH)
print("Rows written:", len(queue))

display(queue.head(20))

Baseline queue written to: baseline_action_score.csv
Rows written: 30000


,rank,content_id,score,reason_code,action
0,1,content_cf56e2e2e282,5,STALE_LOW_CTR,REFRESH
1,2,content_7368877ea310,5,STALE_LOW_CTR,REFRESH
2,3,content_1bfaa38ff26c,5,STALE_LOW_CTR,REFRESH
3,4,content_0a91db491d14,5,STALE_LOW_CTR,REFRESH
4,5,content_5feee3994adb,5,STALE_LOW_CTR,REFRESH
5,6,content_c2d929d83eaa,5,STALE_LOW_CTR,REFRESH
6,7,content_b16bd7307b39,5,STALE_LOW_CTR,REFRESH
7,8,content_fe16a55cd13d,5,STALE_LOW_CTR,REFRESH
8,9,content_ecb6215e79fd,5,STALE_LOW_CTR,REFRESH
9,10,content_928af3e22c80,5,STALE_LOW_CTR,REFRESH


### Top-20 review

The following review examines the highest-ranked items from the baseline queue.

For each item I record the action, reason code, a confidence note, and what could make the recommendation wrong. The confidence notes are qualitative because this is a transparent baseline rather than a fitted probability model.

A high score means that the item matches the rule strongly; it does not mean that the recommended action is guaranteed to improve performance.


In [12]:
# ------------------------------------------------------------
# Top-20 review
# ------------------------------------------------------------

top20 = work.head(20).copy()

top20["confidence_note"] = np.select(
    [
        top20["score"] >= 6,
        top20["score"] >= 4
    ],
    [
        "Higher rule confidence because multiple baseline conditions are met.",
        "Moderate rule confidence because more than one signal contributes."
    ],
    default="Lower rule confidence because the item matches fewer conditions."
)

top20["what_would_make_it_wrong"] = np.select(
    [
        top20["reason_code"] == "STALE_LOW_CTR",
        top20["reason_code"] == "STALE",
        top20["reason_code"] == "LOW_CTR"
    ],
    [
        "Low CTR may reflect search intent, SERP features, or position rather than staleness.",
        "The content may still be performing adequately despite being old.",
        "Low CTR may be caused by search position or a small/unstable impression sample."
    ],
    default="The rule may be too weak to justify action without manual review."
)

review_cols = [
    "rank",
    "score",
    "action",
    "reason_code",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "impressions_90d",
    "confidence_note",
    "what_would_make_it_wrong"
]

print("TOP 20 BASELINE REVIEW")
display(top20[review_cols])

TOP 20 BASELINE REVIEW


,rank,score,action,reason_code,days_since_last_update,ctr,avg_position,impressions_90d,confidence_note,what_would_make_it_wrong
0,1,5,REFRESH,STALE_LOW_CTR,194,0.15,19.7,61678,Moderate rule confidence because more than one...,"Low CTR may reflect search intent, SERP featur..."
1,2,5,REFRESH,STALE_LOW_CTR,194,0.13,24.8,59472,Moderate rule confidence because more than one...,"Low CTR may reflect search intent, SERP featur..."
2,3,5,REFRESH,STALE_LOW_CTR,194,0.23,22.2,25715,Moderate rule confidence because more than one...,"Low CTR may reflect search intent, SERP featur..."
3,4,5,REFRESH,STALE_LOW_CTR,193,0.49,10.5,13299,Moderate rule confidence because more than one...,"Low CTR may reflect search intent, SERP featur..."
4,5,5,REFRESH,STALE_LOW_CTR,194,0.01,39.0,7812,Moderate rule confidence because more than one...,"Low CTR may reflect search intent, SERP featur..."
5,6,5,REFRESH,STALE_LOW_CTR,193,0.20,17.9,7558,Moderate rule confidence because more than one...,"Low CTR may reflect search intent, SERP featur..."
6,7,5,REFRESH,STALE_LOW_CTR,194,0.00,31.0,4590,Moderate rule confidence because more than one...,"Low CTR may reflect search intent, SERP featur..."
7,8,5,REFRESH,STALE_LOW_CTR,194,0.33,16.4,4556,Moderate rule confidence because more than one...,"Low CTR may reflect search intent, SERP featur..."
8,9,5,REFRESH,STALE_LOW_CTR,194,0.38,25.3,4429,Moderate rule confidence because more than one...,"Low CTR may reflect search intent, SERP featur..."
9,10,5,REFRESH,STALE_LOW_CTR,193,0.12,15.8,1697,Moderate rule confidence because more than one...,"Low CTR may reflect search intent, SERP featur..."


### Weak picks and leakage check

The baseline is intentionally simple, so some high-ranked items may be weak picks.

A page can receive a high score because it is old and has low CTR, but that does not prove that refreshing it will improve performance. Low CTR can also reflect search intent, SERP features, ranking position, or limited impression volume.

I also checked the feature construction for leakage. The score does not use `trend_direction`, `trend_pct`, or `is_declining_label`. It also does not use client or content IDs as predictive features. No future-window variables are constructed in this baseline; the rule uses the available 90-day snapshot fields only.

These results should therefore be treated as directional decision-support rather than causal conclusions.


In [13]:
# ------------------------------------------------------------
# Weak-pick check
# ------------------------------------------------------------

print("WEAK PICK CHECK")

weak_candidates = work[
    (
        (work["reason_code"] == "STALE_LOW_CTR")
        & (
            (work["impressions_90d"] < 300)
            | (work["avg_position"] == 0)
        )
    )
    | (
        (work["reason_code"] == "LOW_CTR")
        & (work["impressions_90d"] < 300)
    )
].head(5)

if len(weak_candidates) > 0:
    print("Potential weak picks found:", len(weak_candidates))
    display(
        weak_candidates[
            [
                "rank",
                "score",
                "reason_code",
                "action",
                "days_since_last_update",
                "ctr",
                "avg_position",
                "impressions_90d"
            ]
        ]
    )
else:
    print(
        "No obvious weak pick was found by this automated screen. "
        "The Top-20 should still be inspected manually."
    )

# ------------------------------------------------------------
# Leakage check
# ------------------------------------------------------------

forbidden_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

print("\nLEAKAGE CHECK")

for col in forbidden_columns:
    print(f"{col}: present={col in work.columns}")

print("\nRule feature checks:")
print("Uses trend_direction:", False)
print("Uses trend_pct:", False)
print("Uses is_declining_label:", False)
print("Uses content_id as feature:", False)
print("Uses client_id as feature:", False)

print("\nOutput file exists:", __import__("os").path.exists(OUTPUT_PATH))

WEAK PICK CHECK
Potential weak picks found: 5


,rank,score,reason_code,action,days_since_last_update,ctr,avg_position,impressions_90d
20,21,5,STALE_LOW_CTR,REFRESH,236,0.00,19.1,285
21,22,5,STALE_LOW_CTR,REFRESH,183,0.75,6.9,268
22,23,5,STALE_LOW_CTR,REFRESH,183,0.00,8.0,265
23,24,5,STALE_LOW_CTR,REFRESH,304,0.49,4.8,206
24,25,5,STALE_LOW_CTR,REFRESH,305,0.00,64.5,202



LEAKAGE CHECK
trend_direction: present=True
trend_pct: present=True
is_declining_label: present=False

Rule feature checks:
Uses trend_direction: False
Uses trend_pct: False
Uses is_declining_label: False
Uses content_id as feature: False
Uses client_id as feature: False

Output file exists: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.